# Analisis Comparativo Nacional: Biodiversidad Forestal y Brecha de Apoyo Ambiental
## HackODS UNAM 2026 — ODS 15: Vida de Ecosistemas Terrestres

---

Este notebook complementa el analisis del cuaderno `003_biodiversidad_plotly` con visualizaciones comparativas a nivel nacional, cruzando aprovechamiento maderable con gestion ambiental para identificar las entidades con mayor desequilibrio entre presion forestal y apoyo institucional.

La pregunta central es: *Que entidades extraen mas madera y al mismo tiempo reciben menos apoyo para sus acciones de conservacion?*

---

In [ ]:
# Importaciones
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Paleta de colores del proyecto
template_style = 'plotly_white'
COLOR_PINO   = '#2E8B57'
COLOR_ENCINO = '#8B4513'
COLOR_OTRAS  = '#DAA520'
COLOR_APOYO  = '#4CAF50'
COLOR_SIN    = '#F44336'
COLOR_MICH   = '#FF6F00'

print('Librerias cargadas.')

In [ ]:
# Carga y preparacion de datos
df_bio     = pd.read_csv('../data/procesados/biodiversidad_especies_ods15.csv')
df_gestion = pd.read_csv('../data/procesados/gestion_ambiental_ods15.csv')

# Limpieza de nombre de entidad en biodiversidad
df_bio['estado'] = df_bio['entidad'].str.replace(r'^[0-9]+\s+', '', regex=True).str.strip()
df_bio['Total_Maderable'] = df_bio['Pino'] + df_bio['Encino'] + df_bio['Otras']

# Excluir la fila nacional
df_bio_estados = df_bio[df_bio['estado'] != 'NAL'].copy()

print(f'Registros biodiversidad (sin Nacional): {len(df_bio_estados)}')
print(f'Registros gestion ambiental: {len(df_gestion)}')

---
## 1. Mapa de Calor: Concentracion de Especies por Entidad

Con un heatmap podemos ver de un vistazo que entidades concentran la extraccion de cada especie.

In [ ]:
# Grafico 1: Heatmap de especies por entidad (Top 15)
df_top15 = df_bio_estados.sort_values('Total_Maderable', ascending=False).head(15)

fig1 = px.imshow(
    df_top15[['Pino', 'Encino', 'Otras']].values,
    labels=dict(x='Especie', y='Entidad', color='Volumen (m3)'),
    x=['Pino', 'Encino', 'Otras'],
    y=df_top15['estado'].tolist(),
    color_continuous_scale='YlGn',
    aspect='auto',
    text_auto='.0f'
)

fig1.update_layout(
    title='Mapa de Calor: Volumen Maderable por Especie y Entidad<br><sup>Top 15 entidades con mayor aprovechamiento forestal</sup>',
    template=template_style,
    height=600,
    yaxis=dict(autorange='reversed')
)

fig1.show()

El Pino domina en Durango, Chihuahua, Mexico y Michoacan. Las entidades del sureste (Tabasco, Campeche, Quintana Roo) concentran su extraccion en "Otras" especies tropicales. Esta diferencia refleja ecosistemas distintos bajo distintos tipos de presion.

---
## 2. Brecha de Apoyo: Porcentaje de Acciones Ambientales Sin Financiamiento

Aqui se observa que tan solas estan las comunidades que intentan cuidar su entorno, ordenadas por nivel de desamparo.

In [ ]:
# Grafico 2: Barras horizontales - % acciones sin apoyo
df_gestion_valid = df_gestion[df_gestion['pct_acciones_sin_apoyo'] >= 0].copy()
df_gestion_sorted = df_gestion_valid.sort_values('pct_acciones_sin_apoyo', ascending=True)

# Colores: resaltar Michoacan y las que superan 70%
colores = [
    COLOR_MICH if 'Michoacán' in e else ('#E57373' if v > 70 else '#90CAF9')
    for e, v in zip(df_gestion_sorted['entidad'], df_gestion_sorted['pct_acciones_sin_apoyo'])
]

fig2 = go.Figure(go.Bar(
    y=df_gestion_sorted['entidad'],
    x=df_gestion_sorted['pct_acciones_sin_apoyo'],
    orientation='h',
    marker_color=colores,
    text=df_gestion_sorted['pct_acciones_sin_apoyo'].round(1).astype(str) + '%',
    textposition='outside'
))

fig2.add_vline(
    x=50, line_dash='dash', line_color='gray',
    annotation_text='50% (umbral)', annotation_position='top right'
)

fig2.update_layout(
    title='Porcentaje de Acciones Ambientales Realizadas SIN Apoyo Gubernamental<br><sup>Todas las entidades - Rojo: >70% sin apoyo | Naranja: Michoacan destacado</sup>',
    template=template_style,
    height=800,
    xaxis_title='% Acciones sin apoyo',
    yaxis_title='',
    xaxis_range=[0, 105]
)

fig2.show()

Michoacan presenta un 80.7% de acciones ambientales sin apoyo, situandose entre las entidades con mayor desproteccion. Entidades como Tlaxcala (93%), Sinaloa (86%) y Veracruz (85%) muestran desamparo aun mayor, pero con menor presion por monocultivo.

---
## 3. Cruce Critico: Extraccion Maderable vs. Brecha de Apoyo

Esta visualizacion de dispersion cruza dos ejes: el volumen total de extraccion maderable con el porcentaje de acciones sin apoyo. Las entidades en el cuadrante superior derecho son las mas vulnerables: mucha extraccion y poco respaldo.

In [ ]:
# Grafico 3: Scatter - Cruce aprovechamiento vs brecha

# Mapeo entre codigos cortos y nombres completos
mapeo_nombres = {
    'AGS': 'Aguascalientes', 'BCN': 'Baja California', 'BCS': 'Baja California Sur',
    'CAM': 'Campeche', 'COL': 'Colima', 'CHS': 'Chiapas', 'CHI': 'Chihuahua',
    'CDM': 'Ciudad de México', 'DGO': 'Durango', 'GTO': 'Guanajuato',
    'GRO': 'Guerrero', 'HGO': 'Hidalgo', 'JAL': 'Jalisco', 'MEX': 'México',
    'MIC': 'Michoacán', 'NAY': 'Nayarit', 'NLN': 'Nuevo León',
    'OAX': 'Oaxaca', 'PUE': 'Puebla', 'QRO': 'Querétaro', 'QTR': 'Quintana Roo',
    'SLP': 'San Luis Potosí', 'SIN': 'Sinaloa', 'SON': 'Sonora',
    'TAB': 'Tabasco', 'TAM': 'Tamaulipas', 'TLA': 'Tlaxcala',
    'VER': 'Veracruz', 'YUC': 'Yucatán', 'ZAC': 'Zacatecas'
}

df_bio_estados['nombre_completo'] = df_bio_estados['estado'].map(mapeo_nombres)

# Cruce por nombre parcial
rows_cruce = []
for _, bio_row in df_bio_estados.iterrows():
    nombre = bio_row.get('nombre_completo', '')
    if pd.isna(nombre):
        continue
    match = df_gestion[df_gestion['entidad'].str.contains(nombre, na=False)]
    if not match.empty:
        gest_row = match.iloc[0]
        if gest_row['pct_acciones_sin_apoyo'] >= 0:
            rows_cruce.append({
                'estado': bio_row['estado'],
                'Total_Maderable': bio_row['Total_Maderable'],
                'pct_sin_apoyo': gest_row['pct_acciones_sin_apoyo'],
                'unidades_con_acciones': gest_row['unidades_con_acciones'],
                'es_michoacan': 'Michoacán' in gest_row['entidad']
            })

df_cruce = pd.DataFrame(rows_cruce)

fig3 = px.scatter(
    df_cruce,
    x='Total_Maderable',
    y='pct_sin_apoyo',
    size='unidades_con_acciones',
    color='es_michoacan',
    color_discrete_map={True: COLOR_MICH, False: '#78909C'},
    hover_name='estado',
    text='estado',
    labels={
        'Total_Maderable': 'Volumen Total Maderable (m3)',
        'pct_sin_apoyo': '% Acciones Sin Apoyo',
        'unidades_con_acciones': 'Unidades con Acciones',
        'es_michoacan': 'Michoacan'
    },
    title='Cruce: Extraccion Maderable vs. Brecha de Apoyo Ambiental<br><sup>Tamano = No. de unidades con acciones | Naranja: Michoacan destacado</sup>'
)

fig3.update_traces(textposition='top center', textfont_size=9)

# Lineas de referencia
fig3.add_hline(y=50, line_dash='dot', line_color='gray', opacity=0.5)
fig3.add_vline(x=df_cruce['Total_Maderable'].median(), line_dash='dot', line_color='gray', opacity=0.5)

# Anotacion de zona critica
fig3.add_annotation(
    x=df_cruce['Total_Maderable'].max() * 0.85,
    y=90,
    text='Zona Critica:<br>Alta extraccion + bajo apoyo',
    showarrow=False,
    font=dict(color=COLOR_SIN, size=11),
    bgcolor='rgba(244,67,54,0.1)',
    bordercolor=COLOR_SIN,
    borderwidth=1
)

fig3.update_layout(
    template=template_style,
    height=600,
    showlegend=False
)

fig3.show()

Michoacan se ubica en una posicion de alta extraccion y alto porcentaje de desamparo, confirmando la hipotesis de vulnerabilidad ecosistemica. Durango, aunque extrae mucho mas, tiene un nivel similar de acciones sin apoyo, pero no enfrenta la presion adicional del monocultivo de aguacate.

---
## 4. Composicion Maderable: Todas las Entidades (Treemap)

Un treemap permite visualizar la jerarquia del volumen maderable de cada entidad con su distribucion por especie en una sola vista.

In [ ]:
# Grafico 4: Treemap de composicion maderable
df_melt = df_bio_estados.melt(
    id_vars=['estado'],
    value_vars=['Pino', 'Encino', 'Otras'],
    var_name='Especie',
    value_name='Volumen'
)
df_melt = df_melt[df_melt['Volumen'] > 0]

fig4 = px.treemap(
    df_melt,
    path=['Especie', 'estado'],
    values='Volumen',
    color='Especie',
    color_discrete_map={'Pino': COLOR_PINO, 'Encino': COLOR_ENCINO, 'Otras': COLOR_OTRAS},
    title='Treemap: Distribucion Nacional del Aprovechamiento Maderable<br><sup>Agrupado por especie y entidad - Tamano proporcional al volumen (m3)</sup>'
)

fig4.update_layout(
    template=template_style,
    height=650
)

fig4.update_traces(textinfo='label+percent parent')

fig4.show()

El Pino domina a nivel nacional con mas del 80% del volumen total. Dentro de esa categoria, Durango, Chihuahua y Michoacan son los principales extractores. Las categorias de Encino y Otras tienen presencia significativa solo en algunas entidades, generalmente del sureste y costa.

---
## 5. Indice de Vulnerabilidad Forestal

Se construye un indicador sintetico que combina:
- Volumen de extraccion (estandarizado)
- Porcentaje de acciones sin apoyo (estandarizado)

Un indice alto significa alta extraccion junto con bajo respaldo, es decir, mayor vulnerabilidad ecosistemica.

In [ ]:
# Grafico 5: Ranking de vulnerabilidad
if len(df_cruce) > 0:
    # Estandarizacion min-max
    df_cruce['z_extraccion'] = (df_cruce['Total_Maderable'] - df_cruce['Total_Maderable'].min()) / \
                                (df_cruce['Total_Maderable'].max() - df_cruce['Total_Maderable'].min())
    df_cruce['z_sin_apoyo']  = (df_cruce['pct_sin_apoyo'] - df_cruce['pct_sin_apoyo'].min()) / \
                                (df_cruce['pct_sin_apoyo'].max() - df_cruce['pct_sin_apoyo'].min())
    
    df_cruce['indice_vulnerabilidad'] = (df_cruce['z_extraccion'] * 0.5) + (df_cruce['z_sin_apoyo'] * 0.5)
    df_rank = df_cruce.sort_values('indice_vulnerabilidad', ascending=True).tail(15)
    
    colores_vuln = [
        COLOR_MICH if m else ('#E57373' if v > 0.6 else '#FFB74D' if v > 0.4 else '#81C784')
        for m, v in zip(df_rank['es_michoacan'], df_rank['indice_vulnerabilidad'])
    ]
    
    fig5 = go.Figure(go.Bar(
        y=df_rank['estado'],
        x=df_rank['indice_vulnerabilidad'],
        orientation='h',
        marker_color=colores_vuln,
        text=df_rank['indice_vulnerabilidad'].round(3),
        textposition='outside'
    ))
    
    fig5.update_layout(
        title='Top 15: Indice de Vulnerabilidad Forestal<br><sup>Combina volumen de extraccion + brecha de apoyo ambiental (0 = bajo riesgo, 1 = alto riesgo)</sup>',
        template=template_style,
        height=550,
        xaxis_title='Indice de Vulnerabilidad (0-1)',
        xaxis_range=[0, 1.1]
    )
    
    fig5.show()
else:
    print('No se pudo generar el cruce de datos. Verifica los archivos CSV.')

---
## Conclusiones del Analisis Comparativo Nacional

| Indicador | Michoacan | Contexto Nacional |
|---|---|---|
| Especie mas extraida | Pino (>80%) | Consistente con el patron nacional |
| Posicion en extraccion | Top 5-7 | Detras de Durango y Chihuahua |
| % Acciones sin apoyo | 80.7% | Por encima del promedio nacional |
| Indice de vulnerabilidad | Alto | Agravado por presion del monocultivo de aguacate |

### Recomendaciones para el ODS 15:
1. Focalizar inversion publica en las entidades del cuadrante critico (alta extraccion + bajo apoyo).
2. Priorizar Michoacan como zona de intervencion urgente, dado el efecto combinado de la deforestacion forestal y la expansion del aguacate.
3. Diseñar mecanismos de financiamiento para las unidades de produccion que ya realizan acciones de conservacion sin apoyo.

---
*Notebook elaborado para HackODS UNAM 2026 — Equipo HuerfanitosUTM*